In [ ]:
import pandas as pd
import json

In [ ]:
movies = pd.read_csv(r'/content/drive/MyDrive/dataset_recsys/tmdb_5000_movies.csv')
credits = pd.read_csv(r'/content/drive/MyDrive/dataset_recsys/tmdb_5000_credits.csv')
# ratings = pd.read_csv(r'/content/drive/MyDrive/dataset_recsys/ratings_small.csv')

In [ ]:
movies = pd.merge(movies, credits, on='title', how='left')
movies.shape

(4809, 23)

In [ ]:
movies.head(5)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...",...,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,206647,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...",...,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,49026,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]",...,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,49529,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [ ]:
movies['original_title'].sample(10)

,original_title
659,The Long Kiss Goodnight
2146,A Very Harold & Kumar Christmas
1824,Viy
4707,Sweet Sweetback's Baadasssss Song
4259,Mutant World
531,The Man from U.N.C.L.E.
1339,John Q
3524,Sunshine State
3968,They Came Together
4791,Stories of Our Lives


In [ ]:
# Identify JSON-like columns
def is_json_column(series):
    try:
        sample = series.dropna().iloc[0]  # Take a non-null sample value
        json.loads(sample)  # Try converting to JSON
        return True
    except:
        return False

json_columns = [col for col in movies.columns if is_json_column(movies[col])]
print("Columns with JSON-like structure:", json_columns)

Columns with JSON-like structure: ['genres', 'keywords', 'production_companies', 'production_countries', 'spoken_languages', 'cast', 'crew']


In [ ]:
def flatten_json_column(movies, column):
    movies[column] = movies[column].apply(lambda x: json.loads(x) if isinstance(x, str) else x)

    # Extract unique keys from JSON objects
    keys = set()
    for row in movies[column].dropna():
        for entry in row:
            keys.update(entry.keys())

    # Create new columns with prefixes
    for key in keys:
        movies[f"{column}_{key}"] = movies[column].apply(lambda x: [entry[key] for entry in x] if isinstance(x, list) else None)

    movies.drop(columns=[column], inplace=True)  # Remove the original JSON column

# Flatten all JSON-like columns in `movies`
for col in json_columns:
    flatten_json_column(movies, col)

In [ ]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 39 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   budget                           4809 non-null   int64  
 1   homepage                         1713 non-null   object 
 2   id                               4809 non-null   int64  
 3   original_language                4809 non-null   object 
 4   original_title                   4809 non-null   object 
 5   overview                         4806 non-null   object 
 6   popularity                       4809 non-null   float64
 7   release_date                     4808 non-null   object 
 8   revenue                          4809 non-null   int64  
 9   runtime                          4807 non-null   float64
 10  status                           4809 non-null   object 
 11  tagline                          3965 non-null   object 
 12  title               

In [ ]:
movies= movies[['id','original_title', 'overview', 'cast_name', 'crew_name','crew_job', 'genres_name', 'keywords_name']]
print(movies.iloc[0])

id                                                            19995
original_title                                               Avatar
overview          In the 22nd century, a paraplegic Marine is di...
cast_name         [Sam Worthington, Zoe Saldana, Sigourney Weave...
crew_name         [Stephen E. Rivkin, Rick Carter, Christopher B...
crew_job          [Editor, Production Design, Sound Designer, Su...
genres_name           [Action, Adventure, Fantasy, Science Fiction]
keywords_name     [culture clash, future, space war, space colon...
Name: 0, dtype: object


In [ ]:
movies.isnull().sum()

,0
id,0
original_title,0
overview,3
cast_name,0
crew_name,0
crew_job,0
genres_name,0
keywords_name,0


In [ ]:
movies.dropna(inplace=True)
movies.shape

<ipython-input-75-0e0910eaa7ce>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies.dropna(inplace=True)


(4806, 8)

In [ ]:
movies["director"] = movies.apply(
    lambda row: [name for name, job in zip(row["crew_name"], row["crew_job"]) if job == "Director"]
    if isinstance(row["crew_name"], list) and isinstance(row["crew_job"], list) else [],
    axis=1
)
movies = movies[['id','original_title', 'overview', 'cast_name', 'genres_name', 'keywords_name', 'director']]
movies.head()

,id,original_title,overview,cast_name,genres_name,keywords_name,director
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Sam Worthington, Zoe Saldana, Sigourney Weave...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Johnny Depp, Orlando Bloom, Keira Knightley, ...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...",[Gore Verbinski]
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Daniel Craig, Christoph Waltz, Léa Seydoux, R...","[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...",[Sam Mendes]
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Christian Bale, Michael Caine, Gary Oldman, A...","[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...",[Christopher Nolan]
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Taylor Kitsch, Lynn Collins, Samantha Morton,...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...",[Andrew Stanton]


In [ ]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())
movies.head(1)

<ipython-input-77-b255592b469e>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies['overview'] = movies['overview'].apply(lambda x:x.split())


,id,original_title,overview,cast_name,genres_name,keywords_name,director
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Sam Worthington, Zoe Saldana, Sigourney Weave...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...",[James Cameron]


In [ ]:
# movies["genres_name"] = movies["genres_name"].apply(lambda x: [genre.replace(" ", "") for genre in x] if isinstance(x, list) else x)
# movies['genres_name'].head(1)

In [ ]:
def collapse(L):
    L1 = []
    for i in L:
        L1.append(i.replace(" ",""))
    return L1

In [ ]:
movies['cast_name'] = movies['cast_name'].apply(collapse)
movies['director'] = movies['director'].apply(collapse)
movies['genres_name'] = movies['genres_name'].apply(collapse)
movies['keywords_name'] = movies['keywords_name'].apply(collapse)

In [ ]:
movies.head()

,id,original_title,overview,cast_name,genres_name,keywords_name,director
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[SamWorthington, ZoeSaldana, SigourneyWeaver, ...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...",[JamesCameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[JohnnyDepp, OrlandoBloom, KeiraKnightley, Ste...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...",[GoreVerbinski]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[DanielCraig, ChristophWaltz, LéaSeydoux, Ralp...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...",[SamMendes]
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[ChristianBale, MichaelCaine, GaryOldman, Anne...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...",[ChristopherNolan]
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[TaylorKitsch, LynnCollins, SamanthaMorton, Wi...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...",[AndrewStanton]


In [ ]:
movies['tags'] = movies['overview'] + movies['genres_name'] + movies['keywords_name'] + movies['cast_name']
movies = movies[['id','original_title','director', 'tags']]
movies.head()

,id,original_title,director,tags
0,19995,Avatar,[JamesCameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,[GoreVerbinski],"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,[SamMendes],"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,[ChristopherNolan],"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,[AndrewStanton],"[John, Carter, is, a, war-weary,, former, mili..."


In [ ]:
import nltk
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()

In [ ]:
def stem(text):
    y = []

    # Ensure text is a string before applying `.split()`
    if isinstance(text, list):
        text = " ".join(text)  # Convert list to space-separated string

    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)

movies["tags"] = movies["tags"].apply(stem)

In [ ]:
movies["tags"][3]

"follow the death of district attorney harvey dent, batman assum respons for dent' crime to protect the late attorney' reput and is subsequ hunt by the gotham citi polic department. eight year later, batman encount the mysteri selina kyle and the villain bane, a new terrorist leader who overwhelm gotham' finest. the dark knight resurfac to protect a citi that ha brand him an enemy. action crime drama thriller dccomic crimefight terrorist secretident burglar hostagedrama timebomb gothamc vigilant cover-up superhero villai tragichero terror destruct catwoman catburglar imax flood criminalunderworld batman christianbal michaelcain garyoldman annehathaway tomhardi marioncotillard josephgordon-levitt morganfreeman cillianmurphi junotempl liamneeson matthewmodin alonaboutboul benmendelsohn nestorcarbonel joshpenc tomconti joeyk warrenbrown danielsunjata samkennard aliashtepina nickjulian mirandanolan clairejulien aidangillen burngorman brettcullen reggiele josephlyletaylor chriselli duanehen

In [ ]:
movies['tags'] = movies['tags'].apply(lambda x:x.lower())
movies.head(2)

,id,original_title,director,tags
0,19995,Avatar,[JamesCameron],"in the 22nd century, a parapleg marin is dispa..."
1,285,Pirates of the Caribbean: At World's End,[GoreVerbinski],"captain barbossa, long believ to be dead, ha c..."


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english')

vector = cv.fit_transform(movies['tags']).toarray()

In [ ]:
vector.shape

(4806, 5000)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vector)
similarity.shape

(4806, 4806)

In [ ]:
print(similarity)

[[1.         0.06673261 0.07733089 ... 0.04357102 0.         0.        ]
 [0.06673261 1.         0.07396705 ... 0.02083786 0.         0.02174427]
 [0.07733089 0.07396705 1.         ... 0.02414726 0.         0.        ]
 ...
 [0.04357102 0.02083786 0.02414726 ... 1.         0.04307305 0.04259177]
 [0.         0.         0.         ... 0.04307305 1.         0.08989331]
 [0.         0.02174427 0.         ... 0.04259177 0.08989331 1.        ]]


In [ ]:
similarity_df = pd.DataFrame(similarity)
print(similarity_df)

          0         1         2         3         4         5         6     \
0     1.000000  0.066733  0.077331  0.077822  0.170833  0.091135  0.036197   
1     0.066733  1.000000  0.073967  0.044662  0.061276  0.087171  0.017311   
2     0.077331  0.073967  1.000000  0.051755  0.071007  0.060609  0.020060   
3     0.077822  0.044662  0.051755  1.000000  0.028583  0.073193  0.048450   
4     0.170833  0.061276  0.071007  0.028583  1.000000  0.117156  0.049855   
...        ...       ...       ...       ...       ...       ...       ...   
4801  0.018493  0.035377  0.040996  0.061884  0.152828  0.072471  0.028784   
4802  0.053916  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
4803  0.043571  0.020838  0.024147  0.029161  0.020004  0.017075  0.000000   
4804  0.000000  0.000000  0.000000  0.046159  0.084440  0.054056  0.035783   
4805  0.000000  0.021744  0.000000  0.076073  0.041748  0.035635  0.000000   

          7         8         9     ...      4796      4797    

In [ ]:
print(similarity[0])  # Similarity of movie at index 0 with all others

[1.         0.06673261 0.07733089 ... 0.04357102 0.         0.        ]


In [ ]:
def get_similar_movies(movie_name, top_n):
    """
    Returns the top N most similar movies for a given movie title.

    Parameters:
    - movie_name (str): Name of the movie to compare.
    - top_n (int): Number of similar movies to return.

    Returns:
    - DataFrame with movie titles and similarity scores.
    """
    # Get the movie index from the title
    movie_index = movies[movies["original_title"] == movie_name].index

    if movie_index.empty:
        return f"Movie '{movie_name}' not found in dataset."

    movie_index = movie_index[0]  # Get the first match

    # Get similarity scores for the given movie
    similar_movies = list(enumerate(similarity[movie_index]))

    # Sort by similarity (excluding itself)
    sorted_similar_movies = sorted(similar_movies, key=lambda x: x[1], reverse=True)[1:top_n+1]

    # Create a DataFrame for better visualization
    return pd.DataFrame({
        "Movie Title": [movies.loc[index, "original_title"] for index, _ in sorted_similar_movies],
        "Similarity Score": [score for _, score in sorted_similar_movies]
    })

# Example usage
get_similar_movies("Harry Potter and the Order of the Phoenix", 5)

,Movie Title,Similarity Score
0,Harry Potter and the Goblet of Fire,0.542782
1,Harry Potter and the Half-Blood Prince,0.525596
2,Harry Potter and the Chamber of Secrets,0.512471
3,Harry Potter and the Prisoner of Azkaban,0.482615
4,Harry Potter and the Philosopher's Stone,0.470063


In [ ]:
print(movies.iloc[0])

,0
id,19995
original_title,Avatar
director,[JamesCameron]
tags,"in the 22nd century, a parapleg marin is dispa..."


In [ ]:
def recommend_by_director(director_name, top_n=5):
    """
    Returns the top N movies directed by the given director.

    Parameters:
    - director_name (str): Name of the director.
    - top_n (int, optional): Number of movies to return (default is 5).

    Returns:
    - DataFrame with recommended movie titles.
    """
    # Filter movies by director
    filtered_movies = movies[movies["director"].apply(lambda x: director_name in x)]

    if filtered_movies.empty:
        return f"No movies found for director '{director_name}'."

    # Select top N movies (or all if fewer than top_n exist)
    return filtered_movies[["original_title"]].head(top_n)

# Example usage
recommend_by_director("ChristopherNolan", 5)

,original_title
3,The Dark Knight Rises
65,The Dark Knight
95,Interstellar
96,Inception
119,Batman Begins
